# SwinIR ×3 — GAN 없이 Transformer 로

흐린 위성사진(10 m) → 3배 선명하게(3.33 m).

앞의 세 모델과 달리 **판별자가 없다.** 손실은 L1 하나뿐이라 붕괴할 것도 없다.

| | 구조 | 손실 |
|---|---|---|
| EDSR | CNN (residual) | L1 |
| SRGAN | CNN + GAN | MSE + VGG + 적대적 |
| ESRGAN | CNN (RRDB) + GAN | L1 + VGG + RaGAN |
| **SwinIR** | **Transformer (Swin)** | **L1** |

## 1. 데이터

In [ ]:
import sys, urllib.request
!pip install -q timm

LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'swinir_arch.py', 'swinir_models.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)
    sys.modules.pop(m[:-3], None)     # 이미 불러온 옛 모듈이 남아 있으면 비운다

from sr_utils import *

val_lr, val_hr = pair('validation', REP['validation'])
test_lr = load_test()
show([('validation (Paris)', val_lr, val_hr), ('test (Incheon)', test_lr, None)])

## 2. 훈련

**코드가 도는지 확인하는 용도다.** 적은 데이터로 몇 번만 돌린다.
아래 결과는 전체 데이터로 학습해둔 가중치를 쓴다.

판별자가 없어서 루프가 짧다. 예측하고, L1 손실을 재고, 갱신하는 것이 전부다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from swinir_models import build_swinir

N_TRAIN, EPOCHS, BATCH = 16, 3, 4
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
to_t = lambda a: torch.from_numpy(np.stack(a).transpose(0, 3, 1, 2)).float() / 255
loader = DataLoader(TensorDataset(to_t(lo), to_t(hi)), batch_size=BATCH, shuffle=True)

net = build_swinir(3).to(dev).train()
opt = torch.optim.Adam(net.parameters(), 2e-4, betas=(0.9, 0.99))
crit = nn.L1Loss()

for ep in range(1, EPOCHS + 1):
    tot = 0.0
    for x, y in loader:
        loss = crit(net(x.to(dev)), y.to(dev))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        tot += loss.item()
    print(f'epoch {ep}/{EPOCHS}   L1 {tot/len(loader):.5f}')

## 3. 학습 로그 — 흔들림이 없다

전체 데이터로 100 epoch 돌린 기록이다. GAN 이 없어 손실이 단조롭게 내려간다.
SRGAN·ESRGAN 의 판별자 곡선과 비교해 보면 차이가 뚜렷하다.

In [ ]:
import pandas as pd

MODEL = f'{BASE}/models/04_swinir_x3'
e = pd.read_csv(fetch(f'{MODEL}/statistics/train_results.csv', 'log.csv'), index_col=0)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(e.index, e.l1, color='#2f6f9f', lw=1.6)
ax[0].set_title('Training L1 loss'); ax[0].set_ylabel('L1')

ax[1].plot(e.index, e.PSNR, color='#4f9d69', lw=1.6)
ax[1].set_title('Validation PSNR during training'); ax[1].set_ylabel('dB')

ax[2].step(e.index, e.lr, color='#c96a5b', lw=1.6, where='post')
ax[2].set_yscale('log'); ax[2].set_title('Learning rate (halved 3 times)')
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'L1  {e.l1.iloc[0]:.5f} -> {e.l1.iloc[-1]:.5f}   ({(1-e.l1.iloc[-1]/e.l1.iloc[0])*100:.0f}% 감소)')
print('판별자가 없으니 붕괴도 없다. 학습률이 절반씩 떨어질 때마다 한 단계씩 내려간다.')

## 4. 결과

In [ ]:
from swinir_models import load_swinir

net = load_swinir(fetch(f'{MODEL}/checkpoints/swinir_x3.pth', 'swinir_x3.pth'))
print(f'SwinIR classical  {sum(p.numel() for p in net.parameters())/1e6:.2f}M')

@torch.no_grad()
def upscale(lr):
    t = torch.from_numpy(lr.transpose(2, 0, 1)).float()[None].to(next(net.parameters()).device) / 255
    return (net(t).clamp(0, 1)[0].cpu().numpy().transpose(1, 2, 0) * 255).round().astype('uint8')

zoom([('Original LR', nearest(val_lr)), ('Bicubic', bicubic(val_lr)),
      ('SwinIR', upscale(val_lr)), ('Target HR', val_hr)],
     title='validation (Paris), x3')

## 5. 평가

In [ ]:
rows = compare(upscale, label='SwinIR')

## 6. 최종 테스트 — 인천

정답이 없는 실제 Sentinel-2 촬영본이다. 점수는 못 내고 눈으로 확인한다.

In [ ]:
t_bic = bicubic(test_lr)
t_sr = upscale(test_lr)

zoom([('Original LR', nearest(test_lr)), ('Bicubic', t_bic), ('SwinIR', t_sr)],
     ref=t_bic, title='test (Incheon), x3 - no target')

def sharpness(a):
    return float(cv2.Laplacian(cv2.cvtColor(a, cv2.COLOR_RGB2GRAY), cv2.CV_64F).std())

print(f'{"":10s}{"sharpness":>11s}{"mean RGB":>22s}')
print(f'{"Bicubic":10s}{sharpness(t_bic):11.2f}{str(t_bic.reshape(-1,3).mean(0).round(1)):>22s}')
print(f'{"SwinIR":10s}{sharpness(t_sr):11.2f}{str(t_sr.reshape(-1,3).mean(0).round(1)):>22s}')

imageio.imwrite('incheon_swinir.png', t_sr)
print('\nincheon_swinir.png 저장')

## 7. 네 모델 정리

같은 검증 10패치, 같은 인천 사진으로 잰 값이다.

| 모델 | PSNR | SSIM | 인천 선명도 | 파라미터 |
|---|---|---|---|---|
| Bicubic | 18.15 | 0.4805 | 9.83 | — |
| EDSR | 18.97 | 0.5462 | 14.60 | 1.55M |
| SRGAN | 18.30 | 0.5187 | 22.03 | 0.77M |
| ESRGAN | 16.55 | 0.4208 | **41.59** | 5.91M |
| **SwinIR** | **19.04** | **0.5483** | 13.88 | 11.94M |

**지표와 선명도가 반대로 줄 선다.** PSNR 상위인 SwinIR·EDSR 이 선명도는 최하위이고,
PSNR 최하위인 ESRGAN 이 가장 선명하다.

손실 설계 때문이다. L1 만 쓰면 불확실한 고주파를 만드는 것보다 평균으로 뭉개는 쪽이
손실이 작다. ESRGAN 은 지각 손실이 전체의 97% 라 화소 정확도를 버리고 질감을 만든다.

SwinIR 은 EDSR 보다 파라미터가 7.7배인데 PSNR 이득은 0.07 dB 다.
모델을 키우는 것으로는 넘지 못하는 벽이 있다는 뜻이다.